# Compare your Gephi results with NetworkX
Step 1: type your **Gephi** numbers into the cell below (leave `None` for anything you did not measure).
Step 2: run the notebook. It computes the NetworkX values and shows which ones match.

Do not worry if a number differs slightly. The goal is to **explain** each difference.

In [ ]:
# STEP 1: type your Gephi numbers here (numbers only)
gephi = {
    "nodes":                      None,
    "edges":                      None,
    "average degree":             None,   # from Statistics > Average Degree
    "communities (resolution 1)": None,   # from Statistics > Modularity
    "modularity (resolution 1)":  None,
    "diameter (directed)":        None,   # Statistics > Network Diameter, directed ticked
    "avg path length (directed)": None,
    "diameter (undirected)":      None,   # run again with directed unticked
    "avg path length (undirected)": None,
}
# Top 10 products by PageRank in the top-5-community network: write the titles, in order
gephi_top10_pagerank = []

In [ ]:
import networkx as nx, pandas as pd
from sna_helpers import *
nodes, edges, G = load_amazon()
U = G.to_undirected()
n, m = G.number_of_nodes(), G.number_of_edges()

In [ ]:
comms = nx.community.louvain_communities(U, seed=42)
dd, ad, _ = directed_path_stats(G)

ours = {
    "nodes": n,
    "edges": m,
    "average degree": m / n,
    "communities (resolution 1)": len(comms),
    "modularity (resolution 1)": nx.community.modularity(U, comms),
    "diameter (directed)": dd,
    "avg path length (directed)": ad,
    "diameter (undirected)": nx.approximation.diameter(U),
    "avg path length (undirected)": sample_avg_path(U, k=200),
}

In [ ]:
rows = []
for k, v in ours.items():
    g = gephi.get(k)
    if g is None:
        verdict = "(no Gephi value)"
    elif abs(g - v) <= 0.02 * max(abs(v), 1):
        verdict = "match (within 2%)"
    else:
        verdict = "DIFFERENT: explain why"
    rows.append({"measure": k, "Gephi": g, "NetworkX": round(v, 3), "check": verdict})
pd.DataFrame(rows).set_index("measure")

In [ ]:
# PageRank in the top-5-community network
top5 = sorted(comms, key=len, reverse=True)[:5]
S = G.subgraph(set().union(*top5)).copy()
pr = nx.pagerank(S)
ours_top10 = [S.nodes[k]["label"] for k in sorted(pr, key=pr.get, reverse=True)[:10]]
for i, t in enumerate(ours_top10, 1):
    print(i, t[:60])

if gephi_top10_pagerank:
    same = [t for t in ours_top10 if any(t.lower()[:25] == g.lower()[:25] for g in gephi_top10_pagerank)]
    print("\nTitles in both top-10 lists:", len(same))

## Questions to answer in your own words
1. Which numbers matched exactly, and which did not? For each that did not, give **one** reason (sampling, random start, directed vs undirected, different community split, ...).
2. Your community counts differ between Gephi and NetworkX. Run the Louvain cell with `seed=1`, `seed=2`, `seed=3`. What does that tell you about comparing two community results?
3. Gephi's `Average Degree` and the NetworkX `2m/n` differ by a factor of two. Why?
4. Which tool would you use for 5 million nodes, and why?